# 01 · EDA overview
Volumes, coverage, missingness, KPI distributions, diurnal / weekly shape.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)
from networkanalysis.db.database import query_df, table_counts
from networkanalysis.pipeline.features import build_site_feature_table, KPI_DIRECTION, HEADLINE_KPIS

In [ ]:
feat = build_site_feature_table(); feat.shape

In [ ]:
# volumes & missingness
print(query_df("SELECT COUNT(*) hours, COUNT(DISTINCT site_id) sites, MIN(ts_hour) t0, MAX(ts_hour) t1 FROM agg_site_hourly"))
feat[list(HEADLINE_KPIS)].describe().T

In [ ]:
# diurnal shape of the headline KPIs
g = feat.assign(hour=feat.ts_hour.dt.hour).groupby("hour")[list(HEADLINE_KPIS)].mean()
g.plot(subplots=True, layout=(2,3), figsize=(13,6), title="mean KPI by hour of day"); plt.tight_layout()

In [ ]:
# KPI distributions by morphology
fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, k in zip(axes, ["tcp_client_rtt_ms","dl_throughput_mbps","youtube_qoe_mos"]):
    for m, sub in feat.groupby("morphology"):
        sub[k].plot(kind="kde", ax=ax, label=m)
    ax.set_title(k); ax.legend()
plt.tight_layout()

In [ ]:
# serving-cell resolution quality (data-quality metric)
from networkanalysis.pipeline import resolve_serving_cells
r = resolve_serving_cells(drop_fraction=0.3)
print(f"resolved {len(r):,} tests; match rate {r.correct.mean():.1%}; "
      f"low-confidence share {(r.match_confidence<0.5).mean():.1%}")